# B-Free x GlobalForge — Notebook 03: Evaluation (K3)

Đánh giá **3 models** trên **17 in-the-wild subsets** + **standard benchmarks** (AIGCDetect, GenImage, UnivFD, DRCT, Synthbuster, EvalGEN):

| # | Model | Protocol |
|---|-------|----------|
| 1 | Integrated LoRA (output K2) | B-Free family: multi-crop 504 — 5 crops (center + 4 corners), replicate-pad ảnh < 504, **average logits** |
| 2 | B-Free baseline `BFREE_dino2reg4` | Wrapper5crops gốc của repo (replicate + 5-crop ở mức embedding), num_classes=1 → score = logit |
| 3 | GlobalForge released `REM vit_l_a` | **Protocol code thật** (`eval_in_the_wild.py` + `Get_Transforms`): short side ≤ 1296 → center-crop 224; > 1296 → resize 1296 rồi center-crop 224; PNG → JPEG round-trip q100; `Norm(imagenet)` trong model; softmax → log-odds |

Score = `logit_fake − logit_real` (models 1, 2); model 3 quy về logit bằng `log(p_fake/p_real)` để đồng bộ metrics. Metrics: **AUC, bAcc, NLL, ECE, Pd10, EER** (từ `code/utils/dmetrics.py` của repo).

**CO-SPY: `fake_only`** (paper default) — chỉ ảnh fake, bAcc = accuracy trên fake, các metric còn lại NaN.
**Grouped average** (SynthWildx / WildRF / AIGIBench / CO-SPY / BFree): `combine_parent_dataset_average` — trung bình của trung bình các parent group (giống `eval_in_the_wild.py`).

**Yêu cầu Kaggle inputs:**
- `bfree-wheels` — wheel bundle **do notebook 01 tạo** (CPU/T4 online session, Save Version → New Dataset)
- `k2-output` — output notebook 02 (`bfree_globalforge_lora_r16.pth` hoặc `..._best.pth`)
- `bfree-baseline-weights` — thư mục `BFREE_dino2reg4/` (config.yaml + weights .pth) từ https://grip-unina.github.io/B-Free/ (weights table), upload làm Dataset
- `globalforge-code` — thư mục `code/` của GlobalForge (chứa `models/REM.py`), upload làm Dataset
- `globalforge-backbone-vitla` — HF backbone ViT-L của GlobalForge: hoặc chính là thư mục model (`config.json` ở root) hoặc chứa thư mục con `vit_l_a/`, upload làm Dataset
- `globalforge-weights` — checkpoint 13 parts `checkpoint-best.pth.part_*` (~1.25GB total), upload làm Dataset
- `wild-benchmarks` — DATA_ROOT 17 subsets, layout như `eval_in_the_wild.py` (Chameleon, synthwildx/{dalle3,firefly,midjourney_v5}, WildRF/test/{facebook,reddit,twitter}, AIGIBench/{SocialRF,CommunityAI}, CO-SPY-In-the-Wild/{civitai,dalle3,instavibeai,lexica,midjourney}, RRDataset, B-Free, realchain_CD — mỗi cái có `0_real/` + `1_fake/`); standard benchmarks nằm cùng root (`AIGCDetect/`, `GenImage/`, ... cũng `0_real/1_fake`)

> Notebook này chạy **offline hoàn toàn** trên RTX PRO 6000: KHÔNG apt-get, KHÔNG internet.pip (chỉ `--no-index` từ bundle), KHÔNG HF hub download trong runtime.

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import os, sys, glob

WHEELS_DIR = "/kaggle/input/bfree-wheels"
assert os.path.isdir(WHEELS_DIR), (
    f"Wheel bundle not found at {WHEELS_DIR}. "
    "Attach the 'bfree-wheels' Kaggle dataset before running this notebook."
)
wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, "*.whl")))
print(f"Found {len(wheels)} wheels in {WHEELS_DIR}")
assert wheels, "No .whl files found in the wheel bundle."

!pip install --no-index --find-links={WHEELS_DIR} \
    torch==2.8.0+cu128 torchvision==0.23.0+cu128
!pip install --no-index --find-links={WHEELS_DIR} \
    timm==1.0.22 peft==0.15.2 transformers==4.55.4 \
    pandas==2.3.3 numpy==1.26.4 matplotlib==3.11.1 seaborn==0.13.2 \
    scikit-learn scipy pyyaml pillow tqdm safetensors

In [ ]:
REPO_URL = "https://github.com/P-Bao/B-Free.git"
REPO_DIR = "/kaggle/working/B-Free"
BRANCH = "integration/loss-backbone"

import os, sys, glob
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already cloned.")

assert os.path.isfile(os.path.join(REPO_DIR, "code", "networks", "bfree_globalforge_vit.py")), "Clone failed: backbone file missing."
stubs = glob.glob(os.path.join(REPO_DIR, "code", "modules", "*_stub.py"))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"
print("OK: repo cloned on integration/loss-backbone, no stub files (K0 verified).")

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO_DIR, "code"))

import glob
import io
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT
from train_lora import apply_lora_to_backbone
from utils import dmetrics
from utils.normalization import get_list_norm

GF_CODE_DIR = "/kaggle/input/globalforge-code/code"
GF_BACKBONE_INPUT = "/kaggle/input/globalforge-backbone-vitla"
GF_CKPT_PARTS = "/kaggle/input/globalforge-weights"
K2_OUTPUT_DIR = "/kaggle/input/k2-output"
BFREE_WEIGHTS_DIR = "/kaggle/input/bfree-baseline-weights"
DATA_ROOT = "/kaggle/input/wild-benchmarks"

sys.path.append(GF_CODE_DIR)

DEVICE = "cuda:0"
print(f"torch={torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============ Model 1: Integrated LoRA (K2 output) ============

def load_integrated_lora(k2_output_dir, device=DEVICE):
    candidates = ["bfree_globalforge_lora_r16.pth", "bfree_globalforge_lora_r16_best.pth"]
    ckpt_path = next((os.path.join(k2_output_dir, c) for c in candidates
                      if os.path.isfile(os.path.join(k2_output_dir, c))), None)
    assert ckpt_path, f"No K2 checkpoint found in {k2_output_dir} (expected {candidates})."
    print(f"Loading Integrated LoRA from: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ckpt.get("config", {})
    model = BFreeGlobalForgeViT(
        arch=cfg.get("arch", "vit_base_patch14_reg4_dinov2.lvd142m"),
        num_classes=cfg.get("num_classes", 2),
        img_size=cfg.get("img_size", 504),
        pretrained=False, use_lib=True, use_gsr=True, use_dcs=True,
        lib_kernel=cfg.get("lib_kernel", 3), lib_tau=cfg.get("lib_tau", 0.5),
        gsr_window=cfg.get("gsr_window", 3), gsr_mask_prob=cfg.get("gsr_mask_prob", 1.0),
        dcs_tau=cfg.get("dcs_tau", 0.07), lambda_dcs=cfg.get("lambda_dcs", 0.01),
        label_smoothing=cfg.get("label_smoothing", 0.1),
    )
    model = apply_lora_to_backbone(model, r=cfg.get("lora_rank", 16))
    report = model.load_state_dict(ckpt["model"], strict=False)
    print(f"missing={len(report.missing_keys)} unexpected={len(report.unexpected_keys)} "
          f"epoch={ckpt.get('epoch')} val_bAcc={ckpt.get('val_bacc', 'n/a')}")
    assert not report.unexpected_keys, f"Unexpected keys: {report.unexpected_keys[:10]}"
    return model.to(device).eval()

model_integrated = load_integrated_lora(K2_OUTPUT_DIR)
print("Model 1 (Integrated LoRA) loaded.")

In [ ]:
# ============ Model 2: B-Free baseline (BFREE_dino2reg4, protocol repo) ============
import yaml

from networks import get_network, load_weights

with open(os.path.join(BFREE_WEIGHTS_DIR, "config.yaml")) as f:
    bfree_cfg = yaml.safe_load(f)
model_path = os.path.join(BFREE_WEIGHTS_DIR, bfree_cfg["weights_file"])
print(f"B-Free baseline: arch={bfree_cfg['arch']} norm={bfree_cfg['norm_type']} weights={model_path}")
model_bfree = load_weights(get_network(bfree_cfg["arch"]), model_path)
model_bfree = model_bfree.to(DEVICE).eval()
BFREE_NORM = bfree_cfg["norm_type"]
print("Model 2 (B-Free baseline) loaded.")

In [ ]:
# ============ Model 3: GlobalForge released (REM vit_l_a, reassembled 13 parts) ============
GF_CKPT = "/kaggle/working/checkpoint-best.pth"

parts = sorted(glob.glob(os.path.join(GF_CKPT_PARTS, "checkpoint-best.pth.part_*")))
print(f"checkpoint parts: {len(parts)} found")
assert parts, f"No checkpoint parts under {GF_CKPT_PARTS}"
if not os.path.isfile(GF_CKPT) or os.path.getsize(GF_CKPT) < 1_000_000_000:
    with open(GF_CKPT, "wb") as out:
        for p in parts:
            with open(p, "rb") as f:
                while True:
                    chunk = f.read(1024 * 1024 * 64)
                    if not chunk:
                        break
                    out.write(chunk)
print(f"reassembled checkpoint: {os.path.getsize(GF_CKPT) / 1024**3:.2f} GB")

GF_WEIGHTS_ROOT = "/kaggle/working/gf_weights"
os.makedirs(GF_WEIGHTS_ROOT, exist_ok=True)
vit_la_link = os.path.join(GF_WEIGHTS_ROOT, "vit_l_a")
if not os.path.exists(vit_la_link):
    if os.path.isdir(os.path.join(GF_BACKBONE_INPUT, "vit_l_a")):
        src = os.path.join(GF_BACKBONE_INPUT, "vit_l_a")
    elif os.path.isfile(os.path.join(GF_BACKBONE_INPUT, "config.json")):
        src = GF_BACKBONE_INPUT
    else:
        raise AssertionError(
            f"{GF_BACKBONE_INPUT} must contain either a vit_l_a/ subdir or be the HF model dir itself (config.json).")
    os.symlink(src, vit_la_link)
os.environ["GLOBALFORGE_WEIGHTS_DIR"] = GF_WEIGHTS_ROOT

import models.REM as REM


def _infer_lora_rank_from_state_dict(state_dict):
    for key, value in state_dict.items():
        if "lora_A.default.weight" in key:
            return int(value.shape[0])
    return 16


def _infer_module_switches_from_state_dict(state_dict):
    use_lib = any(key.startswith("lib.") for key in state_dict)
    use_gsr = any(key.startswith("gsr.") for key in state_dict)
    return use_lib, use_gsr


def load_released_globalforge(ckpt_path, device=DEVICE):
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = checkpoint.get("model", checkpoint)
    lora_rank = _infer_lora_rank_from_state_dict(state)
    use_lib, use_gsr = _infer_module_switches_from_state_dict(state)
    print(f"GlobalForge released: lora_rank={lora_rank}, use_lib={use_lib}, use_gsr={use_gsr}")
    model = REM.__dict__["REM"](
        mode="vit_l_a", use_lib=use_lib, use_gsr=use_gsr,
        lib_layer=0, gsr_layer=0, lib_kernel=3, lib_tau=0.5,
        gsr_window=3, gsr_mask_prob=1.0,
    )
    model.load_state_dict(state)
    return model.to(device).eval()

model_gf = load_released_globalforge(GF_CKPT)
print("Model 3 (GlobalForge released) loaded.")

### Scoring kernels (theo protocol code thật của từng model)

- **Model 1 (Integrated)**: replicate-pad ảnh < 504 (numpy `edge` pad — tương đương `replicate_wrap` của Wrapper5crops), 5 crops 504 (center + 4 corners), batch forward, **average logits** → score = `l1 − l0`.
- **Model 2 (B-Free baseline)**: nguyên bản repo — feed full ảnh (ToTensor + Normalize theo config), `Wrapper5crops` tự replicate + 5-crop ở mức patch-embedding, mean 5 views; num_classes=1 → score = logit.
- **Model 3 (GlobalForge)**: `short256_center` với `eval_resize_short=1296` (paper) — KHÔNG phải resize 224 trực tiếp; PNG round-trip JPEG q100 (giống `compress_image`); `Norm(imagenet)` đã nằm trong `REM_Model.forward`; softmax → log-odds.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}
GF_EVAL_RESIZE_SHORT = 1296


def list_images(root, limit=None):
    out = [str(p) for p in sorted(Path(root).iterdir())
           if p.is_file() and p.suffix.lower() in IMG_EXT]
    return out[:limit] if limit else out


def replicate_pad_504(img):
    """Replicate-pad ảnh (PIL) để >= 504x504 (tương đương replicate_wrap của Wrapper5crops)."""
    w, h = img.size
    if w >= 504 and h >= 504:
        return img
    arr = np.asarray(img)
    pad_h, pad_w = max(0, 504 - h), max(0, 504 - w)
    if pad_h or pad_w:
        arr = np.pad(arr, ((0, pad_h), (0, pad_w), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
    return img


def five_crop_boxes(img):
    w, h = img.size
    s = 504
    if w < s or h < s:
        raise ValueError(f"image smaller than 504 after padding: {w}x{h}")
    cx, cy = (w - s) // 2, (h - s) // 2
    return [(cx, cy, s, s), (0, 0, s, s), (w - s, 0, s, s), (0, h - s, s, s), (w - s, h - s, s, s)]


@torch.no_grad()
def score_image_integrated(model, img_path, device=DEVICE):
    img = Image.open(img_path).convert("RGB")
    img = replicate_pad_504(img)
    views = torch.stack([T.Compose(get_list_norm("resnet"))(img.crop(b)) for b in five_crop_boxes(img)])
    logits = model(views.to(device))["logits"].float().mean(dim=0)
    return float(logits[1] - logits[0])


@torch.no_grad()
def score_image_bfree(model, img_path, device=DEVICE, norm_type=BFREE_NORM):
    x = T.Compose(get_list_norm(norm_type))(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    out = model(x)
    if out.shape[1] == 1:
        return float(out[0, 0])
    return float(out[0, 1] - out[0, 0])


@torch.no_grad()
def score_image_globalforge(model, img_path, device=DEVICE):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    short = min(w, h)
    if short > GF_EVAL_RESIZE_SHORT:
        scale = GF_EVAL_RESIZE_SHORT / short
        img = TF.resize(img, [round(h * scale), round(w * scale)],
                        interpolation=T.InterpolationMode.BICUBIC)
    img = TF.center_crop(img, [224, 224])
    if img_path.lower().endswith(".png"):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=100, optimize=True)
        buf.seek(0)
        img = Image.open(buf).convert("RGB").copy()
    x = T.functional.to_tensor(img).unsqueeze(0).to(device)
    out = model(x)
    logits = (out["logits"] if isinstance(out, dict) else out).float().clamp(-30, 30)
    p = float(torch.softmax(logits, dim=1)[0, 1])
    return float(np.log(p / max(1.0 - p, 1e-12)))


SCORERS = {
    "Integrated-LoRA": lambda p: score_image_integrated(model_integrated, p),
    "B-Free-baseline": lambda p: score_image_bfree(model_bfree, p, norm_type=BFREE_NORM),
    "GlobalForge-REM": lambda p: score_image_globalforge(model_gf, p),
}
print("scorers ready:", list(SCORERS))

In [ ]:
# ============ Wild subset registry (17 subsets, layout 0_real/1_fake) ============

WILD_SUBSETS = {
    "Chameleon":           ("Chameleon/0_real", "Chameleon/1_fake", False),
    "SynthWildx-DALLE3":   ("synthwildx/dalle3/0_real", "synthwildx/dalle3/1_fake", False),
    "SynthWildx-Firefly":  ("synthwildx/firefly/0_real", "synthwildx/firefly/1_fake", False),
    "SynthWildx-Midj.":    ("synthwildx/midjourney_v5/0_real", "synthwildx/midjourney_v5/1_fake", False),
    "WildRF-FB":           ("WildRF/test/facebook/0_real", "WildRF/test/facebook/1_fake", False),
    "WildRF-Reddit":       ("WildRF/test/reddit/0_real", "WildRF/test/reddit/1_fake", False),
    "WildRF-Twitter":      ("WildRF/test/twitter/0_real", "WildRF/test/twitter/1_fake", False),
    "AIGIBench-SocRF":     ("AIGIBench/SocialRF/0_real", "AIGIBench/SocialRF/1_fake", False),
    "AIGIBench-ComAI":     ("AIGIBench/CommunityAI/0_real", "AIGIBench/CommunityAI/1_fake", False),
    "CO-SPY-Civitai":      ("CO-SPY-In-the-Wild/civitai/0_real", "CO-SPY-In-the-Wild/civitai/1_fake", True),
    "CO-SPY-DALLE3":       ("CO-SPY-In-the-Wild/dalle3/0_real", "CO-SPY-In-the-Wild/dalle3/1_fake", True),
    "CO-SPY-instavibe.ai": ("CO-SPY-In-the-Wild/instavibeai/0_real", "CO-SPY-In-the-Wild/instavibeai/1_fake", True),
    "CO-SPY-Lexica":       ("CO-SPY-In-the-Wild/lexica/0_real", "CO-SPY-In-the-Wild/lexica/1_fake", True),
    "CO-SPY-Midj.v6":      ("CO-SPY-In-the-Wild/midjourney/0_real", "CO-SPY-In-the-Wild/midjourney/1_fake", True),
    "RR-Dataset":          ("RRDataset/0_real", "RRDataset/1_fake", False),
    "BFree-Online":        ("B-Free/0_real", "B-Free/1_fake", False),
    "real-chain":          ("realchain_CD/0_real", "realchain_CD/1_fake", False),
}

STANDARD_BENCHMARKS = {
    "AIGCDetect":  ("AIGCDetect/0_real", "AIGCDetect/1_fake"),
    "GenImage":    ("GenImage/0_real", "GenImage/1_fake"),
    "UnivFD":      ("UnivFD/0_real", "UnivFD/1_fake"),
    "DRCT":        ("DRCT/0_real", "DRCT/1_fake"),
    "Synthbuster": ("Synthbuster/0_real", "Synthbuster/1_fake"),
    "EvalGEN":     ("EvalGEN/0_real", "EvalGEN/1_fake"),
}

assert os.path.isdir(DATA_ROOT), f"DATA_ROOT {DATA_ROOT} does not exist — attach 'wild-benchmarks' dataset."
available = [k for k, (r, f, _) in WILD_SUBSETS.items()
             if os.path.isdir(os.path.join(DATA_ROOT, r)) and os.path.isdir(os.path.join(DATA_ROOT, f))]
missing = [k for k in WILD_SUBSETS if k not in available]
print(f"wild subsets available: {len(available)}/17")
if missing:
    print("[WARN] missing subsets (skipped):", missing)
assert available, "No wild subset folders found under DATA_ROOT."

MAX_IMAGES = None

In [ ]:
import tqdm


def run_subset(model_name, real_dir, fake_dir, data_root, fake_only=False, max_images=None):
    real_paths = list_images(os.path.join(data_root, real_dir), limit=max_images)
    fake_paths = list_images(os.path.join(data_root, fake_dir), limit=max_images)
    if fake_only:
        paths, labels = fake_paths, [1] * len(fake_paths)
    else:
        paths = real_paths + fake_paths
        labels = [0] * len(real_paths) + [1] * len(fake_paths)
    if not paths:
        return None

    scorer = SCORERS[model_name]
    scores, y, n_skipped = [], [], 0
    for path, label in tqdm.tqdm(list(zip(paths, labels)),
                                 desc=f"{model_name}::{os.path.basename(os.path.dirname(fake_dir))}",
                                 leave=False):
        try:
            scores.append(scorer(path))
            y.append(label)
        except (OSError, UnidentifiedImageError, ValueError):
            n_skipped += 1
    if n_skipped:
        print(f"  [WARN] {n_skipped} unreadable images skipped")
    if not scores:
        return None
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y, dtype=int)

    if fake_only:
        return {"Model": model_name, "n_images": len(scores), "n_skipped": n_skipped,
                "AUC": float("nan"), "bAcc": float((scores > 0).mean() * 100.0),
                "NLL": float("nan"), "ECE": float("nan"),
                "Pd10": float("nan"), "EER": float("nan"), "fake_only": True}
    return {"Model": model_name, "n_images": len(scores), "n_skipped": n_skipped,
            "AUC": float(dmetrics.roc_auc_score(y, scores)),
            "bAcc": float(dmetrics.balanced_accuracy_score(y, scores > 0) * 100.0),
            "NLL": float(dmetrics.balanced_nll_binary(y, scores)),
            "ECE": float(dmetrics.balanced_ece_binary(y, scores)),
            "Pd10": float(dmetrics.pd_at_far(y, scores, 0.10) * 100.0),
            "EER": float(dmetrics.calculate_eer2(y, scores) * 100.0),
            "fake_only": False}


all_results = []
for subset, (real_dir, fake_dir, fake_only) in WILD_SUBSETS.items():
    if not (os.path.isdir(os.path.join(DATA_ROOT, real_dir))
            and os.path.isdir(os.path.join(DATA_ROOT, fake_dir))):
        print(f"[skip] {subset}: folders missing")
        continue
    for model_name in SCORERS:
        r = run_subset(model_name, real_dir, fake_dir, DATA_ROOT,
                       fake_only=fake_only, max_images=MAX_IMAGES)
        if r:
            all_results.append({"Benchmark": subset, **r})
            print(f"{subset:20s} | {model_name:16s} | bAcc={r['bAcc']:6.2f} AUC={r['AUC']:.4f} n={r['n_images']}",
                  flush=True)

wild_df = pd.DataFrame(all_results)
display(wild_df)

In [ ]:
# ============ Standard benchmarks (optional — skip nếu chưa attach) ============
std_results = []
for bench, (real_dir, fake_dir) in STANDARD_BENCHMARKS.items():
    if not (os.path.isdir(os.path.join(DATA_ROOT, real_dir))
            and os.path.isdir(os.path.join(DATA_ROOT, fake_dir))):
        print(f"[skip] {bench}: folders missing under {DATA_ROOT}")
        continue
    for model_name in SCORERS:
        r = run_subset(model_name, real_dir, fake_dir, DATA_ROOT,
                       fake_only=False, max_images=MAX_IMAGES)
        if r:
            std_results.append({"Benchmark": bench, **r})
            print(f"{bench:14s} | {model_name:16s} | bAcc={r['bAcc']:6.2f} AUC={r['AUC']:.4f} n={r['n_images']}",
                  flush=True)

std_df = pd.DataFrame(std_results)
if len(std_df):
    display(std_df)

In [ ]:
# ============ Combine + grouped parent average + save ============
def get_parent_dataset_key(result_key):
    if result_key.startswith("SynthWildx-"): return "SynthWildx"
    if result_key.startswith("WildRF-"): return "WildRF"
    if result_key.startswith("AIGIBench-"): return "AIGIBench"
    if result_key.startswith("CO-SPY-"): return "CO-SPY"
    if result_key.startswith("BFree-"): return "BFree"
    return result_key


def combine_parent_dataset_average(df, metric="bAcc"):
    """Trung bình từng parent group, rồi trung bình các group (giống eval_in_the_wild.py)."""
    out = {}
    for model in df["Model"].unique():
        groups = {}
        for subset, value in df[df["Model"] == model].groupby("Benchmark")[metric].last().items():
            groups.setdefault(get_parent_dataset_key(subset), []).append(float(value))
        out[model] = sum(sum(v) / len(v) for v in groups.values()) / len(groups)
    return out


results_df = pd.concat([wild_df, std_df], ignore_index=True)
results_df.to_csv("/kaggle/working/eval_results.csv", index=False)

wild_only = results_df[results_df["Benchmark"].isin(WILD_SUBSETS)]
avg_rows = [{"Model": m, "Benchmark": "Avg B.Acc (wild, parent-avg)", "bAcc": v, "fake_only": False}
            for m, v in combine_parent_dataset_average(wild_only).items()]
results_df = pd.concat([results_df, pd.DataFrame(avg_rows)], ignore_index=True)
results_df.to_csv("/kaggle/working/eval_results.csv", index=False)
print("saved /kaggle/working/eval_results.csv")
display(results_df.pivot_table(index="Benchmark", columns="Model", values="bAcc", dropna=False))

In [ ]:
# ============ Visualization: bar chart + heatmap ============
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = results_df[results_df["Benchmark"] != "Avg B.Acc (wild, parent-avg)"].copy()
piv_bacc = plot_df.pivot_table(index="Benchmark", columns="Model", values="bAcc")

fig, ax = plt.subplots(figsize=(13, max(6, 0.35 * len(piv_bacc))), dpi=130)
piv_bacc.plot(kind="barh", ax=ax)
ax.set_xlabel("bAcc (%)")
ax.set_title("Balanced Accuracy per subset × model")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("/kaggle/working/eval_bacc_bar.png")

fig, ax = plt.subplots(figsize=(9, max(6, 0.35 * len(piv_bacc))), dpi=130)
sns.heatmap(piv_bacc, annot=True, fmt=".1f", cmap="RdYlGn", vmin=50, vmax=100, ax=ax)
ax.set_title("bAcc heatmap (model × subset)")
fig.tight_layout()
fig.savefig("/kaggle/working/eval_bacc_heatmap.png")
print("saved eval_bacc_bar.png + eval_bacc_heatmap.png")

## Evaluation Complete (K3)

Outputs trong `/kaggle/working/`:
- `eval_results.csv` — model × subset × {AUC, bAcc, NLL, ECE, Pd10, EER} + hàng `Avg B.Acc (wild, parent-avg)`
- `eval_bacc_bar.png` — bar chart bAcc per subset × model
- `eval_bacc_heatmap.png` — heatmap model × subset

**K3 checklist:** notebook chạy được · eval results CSV + heatmap + bar chart.

**Tiếp theo:** K4 (loss interaction analysis) — dùng `train_log.csv` từ K2 với `eval/analyze_loss.py` của repo (Pearson/Spearman correlation CE vs DCS + loss dynamics plot).